# Vision Encoder — Grid Tokenizer + ViT-S

Step 2 of `ARCHITECTURE.md` / `TODO.md`: 8x8 non-overlapping patches + learned color-id embedding + learned 2D row/col positional encoding, feeding a from-scratch ViT-S encoder (~10-20M params). See `baseline.ipynb` for the validated RL loop (step 1).

# Imports

In [3]:
import os
from pathlib import Path

ON_KAGGLE = Path("/kaggle/input").exists()
COMP_DIR = Path("/kaggle/input/arc-prize-2026-arc-agi-3") if ON_KAGGLE else Path("../data")
os.environ["OPERATION_MODE"] = "offline"
os.environ["ENVIRONMENTS_DIR"] = str(COMP_DIR / "environment_files")

from arc_agi import Arcade
from arcengine import GameAction, GameState
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

print("Running on Kaggle:", ON_KAGGLE)
print("Environments dir:", os.environ["ENVIRONMENTS_DIR"])

Running on Kaggle: False
Environments dir: ../data/environment_files


# Baseline Code

In [4]:
class RotaryEmbedding(nn.Module):
    def __init__(self, dim: int, max_len: int = 4096, theta: float = 10000.0):
        super().__init__()
        # dim should be the head dimension (d_model / num_heads)
        self.dim = dim
        
        # Compute frequencies
        inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        
        # Generate time steps
        t = torch.arange(max_len, dtype=torch.float32)
        
        # Outer product to get frequency matrix
        freqs = torch.outer(t, self.inv_freq)
        
        # Standard RoPE uses the matrix duplicated for sin/cos pairs
        emb = torch.cat((freqs, freqs), dim=-1)
        
        # Cache cos and sin embeddings
        self.register_buffer("cos_cached", emb.cos(), persistent=False) # [max_len, dim]
        self.register_buffer("sin_cached", emb.sin(), persistent=False)

    def _rotate_half(self, x: torch.Tensor) -> torch.Tensor:
        # Splits the tensor hidden dim in half, rotates one half
        x1 = x[..., :self.dim // 2]
        x2 = x[..., self.dim // 2:]
        return torch.cat((-x2, x1), dim=-1)

    def forward(self, x: torch.Tensor, seq_len: int) -> tuple[torch.Tensor, torch.Tensor]:
        # x shape: [batch_size, num_heads, seq_len, head_dim]
        # Return sliced cos and sin matrices adapted to current sequence length
        return self.cos_cached[:seq_len, :], self.sin_cached[:seq_len, :]

    def apply_rope(self, x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
        # Unsqueeze to align dimensions with x: [1, 1, seq_len, head_dim]
        cos = cos.unsqueeze(0).unsqueeze(1)
        sin = sin.unsqueeze(0).unsqueeze(1)
        
        # RoPE formula: R(x) = x * cos + rotate_half(x) * sin
        return (x * cos) + (self._rotate_half(x) * sin)


class patch_embedding(nn.Module):
    def __init__(self, input_dim, patch_size, output_dim, image_size=64):
        super(patch_embedding, self).__init__()
        self.patch_size = patch_size
        self.grid_size = image_size // patch_size
        self.proj = nn.Conv2d(input_dim, output_dim, kernel_size=patch_size, stride=patch_size)

        # row/col index per token (row-major), consumed by RoPEAttention below.
        # No additive learned PE here: RoPE encodes position inside attention instead.
        rows = torch.arange(self.grid_size).repeat_interleave(self.grid_size)
        cols = torch.arange(self.grid_size).repeat(self.grid_size)
        self.register_buffer("rows", rows, persistent=False)
        self.register_buffer("cols", cols, persistent=False)

    def forward(self, x):
        # x: (batch_size, channels, height, width)
        x = self.proj(x)  # (batch_size, output_dim, grid_size, grid_size)
        x = x.flatten(2).transpose(1, 2)  # (batch_size, num_patches, output_dim), row-major order
        return x


class RoPEAttention(nn.Module):
    """Multi-head self-attention with 2D axial RoPE (replaces nn.MultiheadAttention).

    head_dim is split in half: the first half is rotated using the token's row
    index, the second half using its col index, so relative row and col offsets
    are both encoded through the standard RoPE rotation trick.
    """
    def __init__(self, embed_dim, num_heads, grid_size):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        assert self.head_dim % 4 == 0, "head_dim must be divisible by 4 for 2D axial RoPE"
        self.axial_dim = self.head_dim // 2

        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.out_proj = nn.Linear(embed_dim, embed_dim)

        self.row_rope = RotaryEmbedding(dim=self.axial_dim, max_len=grid_size)
        self.col_rope = RotaryEmbedding(dim=self.axial_dim, max_len=grid_size)

    def _apply_2d_rope(self, x, rows, cols):
        x_row, x_col = x[..., :self.axial_dim], x[..., self.axial_dim:]
        row_cos, row_sin = self.row_rope.cos_cached[rows], self.row_rope.sin_cached[rows]
        col_cos, col_sin = self.col_rope.cos_cached[cols], self.col_rope.sin_cached[cols]
        x_row = self.row_rope.apply_rope(x_row, row_cos, row_sin)
        x_col = self.col_rope.apply_rope(x_col, col_cos, col_sin)
        return torch.cat([x_row, x_col], dim=-1)

    def forward(self, x, rows, cols):
        # x: (batch_size, seq_len, embed_dim)
        batch_size, seq_len, embed_dim = x.shape
        qkv = self.qkv(x).view(batch_size, seq_len, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]  # each (batch_size, num_heads, seq_len, head_dim)

        q = self._apply_2d_rope(q, rows, cols)
        k = self._apply_2d_rope(k, rows, cols)

        attn_out = F.scaled_dot_product_attention(q, k, v)
        attn_out = attn_out.transpose(1, 2).contiguous().view(batch_size, seq_len, embed_dim)
        return self.out_proj(attn_out)


class RoPETransformerEncoderLayer(nn.Module):
    """Drop-in replacement for nn.TransformerEncoderLayer, using RoPEAttention."""
    def __init__(self, embed_dim, num_heads, grid_size, ff_dim=None):
        super().__init__()
        ff_dim = ff_dim or embed_dim * 4
        self.attention = RoPEAttention(embed_dim, num_heads, grid_size)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.GELU(),
            nn.Linear(ff_dim, embed_dim),
        )

    def forward(self, x, rows, cols):
        x = x + self.attention(self.norm1(x), rows, cols)
        x = x + self.mlp(self.norm2(x))
        return x


class Transformer(nn.Module):
    def __init__(self, input_dim, output_dim, num_heads, num_layers, patch_size):
        super(Transformer, self).__init__()
        self.embedding = patch_embedding(input_dim, patch_size, output_dim)
        self.layers = nn.ModuleList([
            RoPETransformerEncoderLayer(output_dim, num_heads, self.embedding.grid_size)
            for _ in range(num_layers)
        ])
        self.fc_out = nn.Linear(output_dim, output_dim)

    def forward(self, x):
        x = self.embedding(x)
        rows, cols = self.embedding.rows, self.embedding.cols
        for layer in self.layers:
            x = layer(x, rows, cols)
        x = self.fc_out(x)
        return x


class VisionEncoding(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(VisionEncoding, self).__init__()
        self.norm1 = nn.LayerNorm(output_dim)
        self.norm2 = nn.LayerNorm(output_dim)
        self.embedding = patch_embedding(input_dim=input_dim, patch_size=4, output_dim=output_dim)
        self.attention = RoPEAttention(embed_dim=output_dim, num_heads=4, grid_size=self.embedding.grid_size)
        self.fc_out = nn.Linear(output_dim, output_dim)

    def forward(self, x):
        x = self.embedding(x)
        rows, cols = self.embedding.rows, self.embedding.cols
        x = self.norm1(x)
        x = self.attention(x, rows, cols)
        x = self.norm2(x)
        x = self.fc_out(x)
        return x

class PolicyNetwork(nn.Module):
    """Wraps the Transformer encoder with a pooled policy head over a fixed action space."""
    def __init__(self, encoder, num_actions):
        super().__init__()
        self.encoder = encoder
        self.policy_head = nn.Linear(encoder.fc_out.out_features, num_actions)

    def forward(self, x):
        tokens = self.encoder(x)          # (batch, num_patches, output_dim)
        pooled = tokens.mean(dim=1)        # (batch, output_dim)
        return self.policy_head(pooled)    # (batch, num_actions)


# SimpleAction ids only (ACTION6 needs (x, y) click coords, not handled here).
ACTION_SPACE = [GameAction.ACTION1, GameAction.ACTION2, GameAction.ACTION3,
                 GameAction.ACTION4, GameAction.ACTION5, GameAction.ACTION7]
ACTION_ID_TO_INDEX = {a.value: i for i, a in enumerate(ACTION_SPACE)}


def frame_to_tensor(frame):
    # frame.frame can carry a variable number of layers across steps (not fixed
    # to 1) -- keep only the base layer so the model's input channel count stays
    # constant.
    grid = frame.frame[0]  # (64, 64), int8 values 0-15
    return torch.from_numpy(grid).float().unsqueeze(0).unsqueeze(0)  # (1, 1, 64, 64)


def select_action(policy, frame):
    x = frame_to_tensor(frame)
    logits = policy(x).squeeze(0)  # (num_actions,)

    mask = torch.full((len(ACTION_SPACE),), float("-inf"))
    for action_id in frame.available_actions:
        if action_id in ACTION_ID_TO_INDEX:
            mask[ACTION_ID_TO_INDEX[action_id]] = 0.0

    dist = torch.distributions.Categorical(logits=logits + mask)
    index = dist.sample()
    return ACTION_SPACE[index.item()], dist.log_prob(index)


# --- ls20-specific reward shaping ------------------------------------------
# Diagnostic-only: hardcoded from inspecting ls20's raw grid (colors/regions
# below don't generalize to the other 24 games). Goal is to prove GRPO can
# learn *something* on this baseline before investing in JEPA/GRPO for real
# (see ARCHITECTURE.md step 1). Not meant to survive into the general agent.
#
# Findings from inspecting frame.frame[0] on ls20:
#   - color 9 draws three separate things: the moving cursor we control, a
#     fixed "target orientation" icon (rows 0-16, cols 31-40), and a fixed
#     "current orientation" icon (rows 53-63, cols 0-11, bottom-left).
#   - colors 0/1 form the small purple cross (~6 px) the cursor must cross.
#   - driving the cursor over the cross visibly rotates the bottom-left icon
#     (confirmed empirically), which is presumably the actual win condition
#     once it matches the target icon -- exact match rule not verified, so
#     only a "rotation happened" bonus is used below, not a shape-similarity
#     potential (a wrong potential can mislead training worse than no shaping).

def locate_player(grid):
    mask = (grid == 9)
    mask[:17, 31:41] = False   # zero out the fixed target-orientation icon
    mask[53:, :12] = False     # zero out the fixed current-orientation icon
    coords = np.argwhere(mask)
    if len(coords) == 0:
        return None
    return coords[:, 0].mean(), coords[:, 1].mean()


def locate_cross(grid):
    coords = np.argwhere((grid == 0) | (grid == 1))
    if len(coords) == 0:
        return None
    return coords[:, 0].mean(), coords[:, 1].mean()


def orientation_icon(grid):
    return grid[54:64, 0:12]  # bottom-left "current orientation" icon


def compute_reward(prev_frame, frame):
    reward = float(frame.levels_completed - prev_frame.levels_completed)
    if frame.state == GameState.WIN:
        reward += 1.0
    elif frame.state == GameState.GAME_OVER:
        reward -= 1.0

    # Potential-based shaping (Ng et al. 1999): dense reward for closing the
    # distance to the cross, doesn't change the optimal policy.
    prev_grid, grid = prev_frame.frame[0], frame.frame[0]
    prev_player, player = locate_player(prev_grid), locate_player(grid)
    cross = locate_cross(grid)
    if prev_player is not None and player is not None and cross is not None:
        prev_dist = abs(prev_player[0] - cross[0]) + abs(prev_player[1] - cross[1])
        dist = abs(player[0] - cross[0]) + abs(player[1] - cross[1])
        reward += 0.02 * (prev_dist - dist)

    # Bonus for triggering a rotation event (cursor passed over the cross).
    if not np.array_equal(orientation_icon(prev_grid), orientation_icon(grid)):
        reward += 0.1

    reward -= 0.01  # step penalty, encourages efficiency
    return reward
# -----------------------------------------------------------------------------


def rollout(env, policy, max_steps=50):
    # Simplification vs ARCHITECTURE.md: reset() is assumed to give a comparable
    # starting state across rollouts (not verified — see "hidden randomness" caveat).
    frame = env.reset()
    log_probs = []
    total_reward = 0.0

    for _ in range(max_steps):
        if frame.state != GameState.NOT_FINISHED:
            break
        prev_frame = frame
        action, log_prob = select_action(policy, frame)
        frame = env.step(action)
        total_reward += compute_reward(prev_frame, frame)
        log_probs.append(log_prob)
        if frame.state != GameState.NOT_FINISHED:
            break

    return log_probs, total_reward


def collect_group(env, policy, group_size, max_steps=50):
    return [rollout(env, policy, max_steps) for _ in range(group_size)]


def grpo_update(rollouts, optimizer):
    rewards = torch.tensor([r for _, r in rollouts])
    advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-8)

    loss = torch.tensor(0.0)
    for (log_probs, _), advantage in zip(rollouts, advantages):
        if not log_probs:
            continue
        loss = loss - advantage * torch.stack(log_probs).sum()
    loss = loss / len(rollouts)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item(), rewards.mean().item()

# New Code

In [5]:
# Game-agnostic reward: overrides the ls20-specific compute_reward from the
# "Baseline Code" section above (distance-to-cross / rotation-bonus shaping).
# That version was hand-tuned on ls20's pixel semantics -- it doesn't
# generalize to the other 24 games, and hand-crafting per-game heuristics
# defeats the point of an agent that learns to play, rather than being told
# how. Only the signal every game actually exposes (levels_completed / WIN /
# GAME_OVER) is used here; the Intuition (JEPA) network is what's supposed to
# carry the agent's understanding of a given game's dynamics, not the reward.
def compute_reward(prev_frame, frame):
    reward = float(frame.levels_completed - prev_frame.levels_completed)
    if frame.state == GameState.WIN:
        reward += 1.0
    elif frame.state == GameState.GAME_OVER:
        reward -= 1.0
    reward -= 0.01  # step penalty, encourages efficiency
    return reward

In [6]:
class Eyes(nn.Module):
    """Vision encoder ("eyes"): wraps the RoPE ViT (from baseline.ipynb) with
    mean-pooling to a single latent vector z."""
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder  # Transformer instance

    def forward(self, x):
        tokens = self.encoder(x)   # (batch, num_patches, latent_dim)
        return tokens.mean(dim=1)  # (batch, latent_dim)


class Brain(nn.Module):
    """From-scratch reasoning transformer ("brain"): takes the eyes latent z_t
    plus N learned role-queries (fixed learned vectors -- not literal tokenized
    text, since there is no pretrained language prior to exploit here; see
    DESIGN_LOG 2026-08-14 for why this is trained, not frozen), and predicts a
    (t_horizon, num_actions) logits grid: one action distribution per future
    timestep, predicted open-loop in a single forward pass.
    """
    def __init__(self, latent_dim, num_actions, t_horizon, num_queries=4, num_heads=4, num_layers=2):
        super().__init__()
        self.t_horizon = t_horizon
        self.num_actions = num_actions
        self.latent_proj = nn.Linear(latent_dim, latent_dim)
        self.queries = nn.Parameter(torch.randn(num_queries, latent_dim) * 0.02)
        layer = nn.TransformerEncoderLayer(d_model=latent_dim, nhead=num_heads, batch_first=True)
        self.transformer = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.action_head = nn.Linear(latent_dim, num_actions * t_horizon)

    def forward(self, z):
        # z: (batch, latent_dim)
        batch_size = z.shape[0]
        z_tok = self.latent_proj(z).unsqueeze(1)                          # (batch, 1, latent_dim)
        query_tok = self.queries.unsqueeze(0).expand(batch_size, -1, -1)  # (batch, num_queries, latent_dim)
        tokens = torch.cat([z_tok, query_tok], dim=1)
        out = self.transformer(tokens)
        pooled = out.mean(dim=1)
        return self.action_head(pooled).view(batch_size, self.t_horizon, self.num_actions)


class Intuition(nn.Module):
    """World model ("intuition"), I-JEPA style: given z_t and the t_horizon
    actions taken, predicts the future latents z_hat_{t+1..t+k} -- trained
    against a stop-grad EMA target encoder's real-frame latents (MSE), not
    against raw pixels, to avoid blurry mean-collapse on multimodal futures.
    """
    def __init__(self, latent_dim, num_actions, t_horizon, num_heads=4, num_layers=2):
        super().__init__()
        self.t_horizon = t_horizon
        self.action_embed = nn.Embedding(num_actions, latent_dim)
        self.step_embed = nn.Embedding(t_horizon, latent_dim)  # disambiguates chunk position
        layer = nn.TransformerEncoderLayer(d_model=latent_dim, nhead=num_heads, batch_first=True)
        self.transformer = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.predict_head = nn.Linear(latent_dim, latent_dim)

    def forward(self, z, action_indices):
        # z: (batch, latent_dim), action_indices: (batch, t_horizon) long
        batch_size = z.shape[0]
        z_tok = z.unsqueeze(1)
        act_tok = self.action_embed(action_indices)
        step_ids = torch.arange(self.t_horizon, device=z.device).unsqueeze(0).expand(batch_size, -1)
        act_tok = act_tok + self.step_embed(step_ids)
        tokens = torch.cat([z_tok, act_tok], dim=1)
        out = self.transformer(tokens)
        return self.predict_head(out[:, 1:, :])  # (batch, t_horizon, latent_dim)


class ARC_AGI_3(nn.Module):
    """Eyes -> Brain (policy, GRPO) + Intuition (world model, JEPA) pipeline.
    See DESIGN_LOG.md 2026-08-14 for the design discussion and open risks
    (open-loop k-action legality, pixel- vs latent-space world model)."""
    def __init__(self, input_dim, latent_dim, patch_size, num_heads, eyes_layers, brain_layers,
                 intuition_layers, num_actions, t_horizon, num_queries=4, ema_decay=0.99):
        super().__init__()
        self.eyes = Eyes(Transformer(input_dim=input_dim, output_dim=latent_dim, num_heads=num_heads,
                                      num_layers=eyes_layers, patch_size=patch_size))
        # EMA target encoder for JEPA -- same architecture, no grad, updated via momentum
        # (ARCHITECTURE.md: "online encoder produces z_t, EMA target encoder produces z_{t+1}").
        self.target_eyes = Eyes(Transformer(input_dim=input_dim, output_dim=latent_dim, num_heads=num_heads,
                                             num_layers=eyes_layers, patch_size=patch_size))
        self.target_eyes.load_state_dict(self.eyes.state_dict())
        for p in self.target_eyes.parameters():
            p.requires_grad_(False)
        self.ema_decay = ema_decay

        self.brain = Brain(latent_dim, num_actions, t_horizon, num_queries=num_queries,
                            num_heads=num_heads, num_layers=brain_layers)
        self.intuition = Intuition(latent_dim, num_actions, t_horizon,
                                    num_heads=num_heads, num_layers=intuition_layers)

    @torch.no_grad()
    def update_target(self):
        for p_online, p_target in zip(self.eyes.parameters(), self.target_eyes.parameters()):
            p_target.mul_(self.ema_decay).add_(p_online, alpha=1 - self.ema_decay)

    def forward(self, x):
        z = self.eyes(x)        # (batch, latent_dim)
        logits = self.brain(z)  # (batch, t_horizon, num_actions)
        return z, logits

    def imagine(self, z, action_indices):
        return self.intuition(z, action_indices)  # (batch, t_horizon, latent_dim)

    @torch.no_grad()
    def target_latent(self, x):
        return self.target_eyes(x)  # stop-grad by construction (no_grad + frozen params)

In [ ]:
def rollout_chunk(env, model, t_horizon, max_chunks=6, curiosity_coef=0.0):
    """One episode, decided in open-loop chunks of t_horizon actions at a time
    (the brain predicts all t actions from a single frame, before seeing any
    of the t-1 intermediate frames).

    DESIGN_LOG 2026-08-14 risk #3: only the first action's legality is known
    at decision time; a later action can turn out illegal once we get there.
    Mitigation used here: fall back to a uniformly-random *legal* action, and
    recompute its log_prob under the same brain distribution -- so GRPO's
    credit assignment always matches the action actually executed, not the
    one originally sampled.

    Also records the full trajectory (grid + action + reward per real step,
    starting from the reset frame) so the best episode can be replayed later
    with render_trajectory().
    """
    frame = env.reset()
    chunks = []
    trajectory = [{"grid": frame.frame[0].copy(), "action": None, "reward": 0.0}]
    for _ in range(max_chunks):
        if frame.state != GameState.NOT_FINISHED:
            break
        x = frame_to_tensor(frame)
        z, logits = model(x)
        logits = logits.squeeze(0)  # (t_horizon, num_actions)

        mask = torch.zeros(t_horizon, len(ACTION_SPACE))
        first_mask = torch.full((len(ACTION_SPACE),), float("-inf"))
        for action_id in frame.available_actions:
            if action_id in ACTION_ID_TO_INDEX:
                first_mask[ACTION_ID_TO_INDEX[action_id]] = 0.0
        mask[0] = first_mask  # only step 0's availability is known ahead of time

        dist = torch.distributions.Categorical(logits=logits + mask)
        # Per-slot entropy of the action distribution, kept with its grad so
        # an exploration bonus can be added at update time (see compute_losses
        # and DESIGN_LOG 2026-08-16: policy collapsed around step 200-230 in
        # the first joint run and never recovered -- no entropy term meant no
        # gradient pressure was left to pull it back out).
        entropy = dist.entropy()  # (t_horizon,)
        action_indices = dist.sample()
        executed_indices = action_indices.tolist()

        extrinsic_reward = 0.0  # task-only reward (levels_completed/WIN/GAME_OVER/step penalty)
        prev_frame = frame
        real_frames = []
        for i in range(t_horizon):
            if frame.state != GameState.NOT_FINISHED:
                break
            step_idx = executed_indices[i]
            action_id = ACTION_SPACE[step_idx].value
            if action_id not in frame.available_actions:
                legal = [a for a in frame.available_actions if a in ACTION_ID_TO_INDEX]
                if not legal:
                    break
                step_idx = ACTION_ID_TO_INDEX[random.choice(legal)]
                executed_indices[i] = step_idx
            action = ACTION_SPACE[step_idx]
            frame = env.step(action)
            real_frames.append(frame_to_tensor(frame))
            step_reward = compute_reward(prev_frame, frame)
            extrinsic_reward += step_reward
            trajectory.append({"grid": frame.frame[0].copy(), "action": action, "reward": step_reward})
            prev_frame = frame

        # Curiosity bonus (generic, no game-specific knowledge): reward is
        # boosted by how *wrong* Intuition's own prediction was for the frames
        # actually visited. High surprise -> more reward -> policy is nudged
        # toward states the world model doesn't understand yet, instead of
        # pure random walk. Self-annealing: as Intuition learns a region well,
        # jepa error there drops, so the bonus for revisiting it fades too.
        # Uses only the model's own prediction error, not pixel/color
        # heuristics -- doesn't violate the no-game-hacking rule.
        #
        # Kept separate from extrinsic_reward (DESIGN_LOG 2026-08-16): a first
        # attempt mixed the two into one "reward" used both for GRPO advantage
        # *and* for picking/reporting the "best" episode -- with curiosity_coef
        # too high, the intrinsic term dwarfed the sparse task signal (~5 vs
        # ~1), so "best episode" ended up ranking pure novelty-seeking above
        # actual task progress (a real level completion was never reached, the
        # reported best episode was picked solely for being "surprising").
        curiosity_bonus = 0.0
        if real_frames and curiosity_coef > 0:
            with torch.no_grad():
                preds = model.imagine(z, torch.tensor(executed_indices).unsqueeze(0))
                targets = torch.cat([model.target_latent(f) for f in real_frames], dim=0)
                n_real = targets.shape[0]
                surprise = F.mse_loss(preds[0, :n_real], targets, reduction="none").mean(dim=-1)
            curiosity_bonus = curiosity_coef * surprise.sum().item()
        total_reward = extrinsic_reward + curiosity_bonus

        # Clamped at the source (not just via grad clipping downstream): the
        # illegal-action fallback above can force-execute an action the
        # current (possibly narrow) policy assigns near-zero probability to,
        # sending log_prob -> -inf and grpo_loss's magnitude with it (observed
        # -64 -> -1688 growth, DESIGN_LOG 2026-08-16 risk #3). Gradient
        # clipping only bounds the *norm* of the resulting update, so a huge
        # grpo_loss still dominates the clipped gradient's *direction* -- e.g.
        # it swamped the self-imitation replay signal in the first A/B test
        # (grpo_loss ~-350 vs replay_coef*replay_loss ~3.5). Clamping keeps
        # grpo_loss in the same ballpark as the other loss terms.
        executed_log_probs = dist.log_prob(torch.tensor(executed_indices)).clamp(min=-5.0)
        # x is stored (plain tensor, no autograd graph attached) so this chunk
        # can be replayed later through a *future* version of the model --
        # see update_replay_buffer / self_imitation_loss below.
        chunks.append(dict(z=z, action_indices=torch.tensor(executed_indices),
                            log_probs=executed_log_probs, reward=total_reward,
                            extrinsic_reward=extrinsic_reward, real_frames=real_frames,
                            entropy=entropy, x=x))
        if frame.state != GameState.NOT_FINISHED:
            break
    return chunks, trajectory


def collect_group(env, model, group_size, t_horizon, max_chunks=6, curiosity_coef=0.0):
    return [rollout_chunk(env, model, t_horizon, max_chunks, curiosity_coef) for _ in range(group_size)]


def compute_losses(rollouts, model):
    # GRPO: group-relative advantage over each *episode's* total reward
    # (summed across all its chunks), same mechanism as baseline.ipynb.
    # rollouts: list of (chunks, trajectory) tuples, see rollout_chunk().
    episode_rewards = torch.tensor([sum(c["reward"] for c in chunks) for chunks, _ in rollouts])
    advantages = (episode_rewards - episode_rewards.mean()) / (episode_rewards.std() + 1e-8)

    grpo_loss = torch.tensor(0.0)
    jepa_loss = torch.tensor(0.0)
    entropy_sum = torch.tensor(0.0)
    n_jepa_terms = 0
    n_entropy_terms = 0

    for (chunks, _), advantage in zip(rollouts, advantages):
        for c in chunks:
            grpo_loss = grpo_loss - advantage * c["log_probs"].sum()
            entropy_sum = entropy_sum + c["entropy"].sum()
            n_entropy_terms += c["entropy"].numel()

            if c["real_frames"]:
                preds = model.imagine(c["z"], c["action_indices"].unsqueeze(0))  # (1, t_horizon, latent_dim)
                with torch.no_grad():
                    targets = torch.cat([model.target_latent(f) for f in c["real_frames"]], dim=0)
                n_real = targets.shape[0]  # chunk may have ended early (episode terminated mid-chunk)
                jepa_loss = jepa_loss + F.mse_loss(preds[0, :n_real], targets)
                n_jepa_terms += 1

    grpo_loss = grpo_loss / len(rollouts)
    jepa_loss = jepa_loss / max(n_jepa_terms, 1)
    entropy_mean = entropy_sum / max(n_entropy_terms, 1)  # mean entropy per predicted action slot
    return grpo_loss, jepa_loss, entropy_mean


def update_replay_buffer(buffer, rollouts, capacity=8):
    """Self-imitation buffer (Oh et al. 2018, "Self-Imitation Learning"):
    keeps the top-`capacity` chunks ever seen, ranked by extrinsic (task)
    reward -- so a rare lucky success survives past the single update it was
    found in, instead of being thrown away and resampled fresh next epoch.

    DESIGN_LOG 2026-08-16: the H1-H4 ablation showed plain GRPO finds the
    *same single* win whether given 50 or 500 epochs (1/500 epochs ever beat
    baseline either way) -- more budget or a bigger curiosity bonus didn't
    help, because nothing ever revisits a success once its one gradient step
    is spent. This buffer is the fix: it keeps replaying known-good chunks
    so the policy has to actively unlearn them to lose the signal.

    Stores raw frames (`x`) + executed actions, not z/log_probs, which go
    stale the moment the model's weights change.
    """
    for chunks, _ in rollouts:
        for c in chunks:
            buffer.append({"x": c["x"], "action_indices": c["action_indices"],
                            "extrinsic_reward": c["extrinsic_reward"]})
    buffer.sort(key=lambda e: e["extrinsic_reward"], reverse=True)
    del buffer[capacity:]
    return buffer


def self_imitation_loss(model, buffer, replay_batch=4):
    """Auxiliary loss: re-run a few of the best-ever chunks through the
    *current* model (fresh forward pass, current weights) and push their
    log_prob back up -- imitate what already worked, rather than relying on
    GRPO alone to rediscover (and likely lose again) the same success.
    No step-0 action-legality mask is reapplied here (unlike rollout_chunk):
    we're not sampling a new action, only scoring the log_prob of one we
    already know was legal and executed. Clamped for the same reason as
    rollout_chunk's log_probs -- a replayed action can look very unlikely to
    a policy that has since drifted away from it.
    """
    if not buffer:
        return torch.tensor(0.0)
    sample = random.sample(buffer, min(replay_batch, len(buffer)))
    loss = torch.tensor(0.0)
    for entry in sample:
        _, logits = model(entry["x"])
        dist = torch.distributions.Categorical(logits=logits.squeeze(0))
        loss = loss - dist.log_prob(entry["action_indices"]).clamp(min=-5.0).sum()
    return loss / len(sample)


def update_best_run(best_run, rollouts):
    """Tracks the single best-reward episode seen across all training rollouts
    (any epoch so far), so it can be replayed afterwards with
    render_trajectory() to inspect what the agent actually does.

    Ranked by extrinsic (task) reward only, not extrinsic+curiosity -- see
    DESIGN_LOG 2026-08-16: ranking on the mixed total let a purely
    novelty-seeking episode (no real task progress) look like the best one.
    """
    for chunks, trajectory in rollouts:
        extrinsic_total = sum(c["extrinsic_reward"] for c in chunks)
        if extrinsic_total > best_run["reward"]:
            best_run["reward"] = extrinsic_total
            best_run["total_reward"] = sum(c["reward"] for c in chunks)  # informational only
            best_run["trajectory"] = trajectory
    return best_run


def render_trajectory(trajectory, interval=300):
    """Plays back a trajectory (list of {'grid', 'action', 'reward'}, as
    produced by rollout_chunk) as an inline scrubbable animation -- call on
    best_run["trajectory"] to watch the agent play and inspect its behavior."""
    from matplotlib.animation import FuncAnimation
    from IPython.display import HTML

    fig, ax = plt.subplots()
    im = ax.imshow(trajectory[0]["grid"], vmin=0, vmax=15)
    ax.axis("off")
    title = ax.set_title("step 0 (reset)")

    def label(entry, i):
        if entry["action"] is None:
            return f"step {i} (reset)"
        return f"step {i} | action {entry['action'].name} | reward {entry['reward']:+.3f}"

    def update(i):
        entry = trajectory[i]
        im.set_data(entry["grid"])
        title.set_text(label(entry, i))
        return im, title

    anim = FuncAnimation(fig, update, frames=len(trajectory), interval=interval, blit=False)
    plt.close(fig)
    return HTML(anim.to_jshtml())


arc = Arcade()
env = arc.make("ls20")
frame = env.reset()

# input_dim, latent_dim, patch_size, num_heads, eyes_layers, brain_layers, intuition_layers, num_actions, t_horizon
model = ARC_AGI_3(input_dim=1, latent_dim=16, patch_size=4, num_heads=2, eyes_layers=1,
                   brain_layers=1, intuition_layers=1, num_actions=len(ACTION_SPACE),
                   t_horizon=4, num_queries=4)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 50    # capped short run to check the gradient-clipping fix before committing to 500
group_size = 24  # G episodes per GRPO update -- raised from 8 so a group is more
                 # likely to contain both a success and a failure (needed for
                 # nonzero group-relative advantage on this sparse-reward game)
t_horizon = 4    # actions predicted open-loop per brain call (kept small, see DESIGN_LOG risk #3)
max_chunks = 12  # up to max_chunks * t_horizon real env steps per episode
jepa_weight = 1.0
entropy_coef = 0.001  # exploration bonus weight -- lowered from 0.01: at that value it was
                       # the only nonzero term whenever grpo_loss hit exactly 0 (all-identical
                       # group rewards), permanently pinning the policy near max entropy and
                       # preventing rare successes from ever being reinforced (see DESIGN_LOG
                       # 2026-08-16 zero-variance trap).
curiosity_coef = 0#0.005  # intrinsic reward weight on Intuition's own prediction error (see
                        # rollout_chunk) -- lowered from 0.05: at that value the intrinsic bonus
                        # (~5/episode early on) dwarfed the sparse task reward (max ~1), so the
                        # policy was being reinforced for chasing novelty, not task progress
                        # (see DESIGN_LOG 2026-08-16). Meant as a small nudge, not the dominant
                        # term -- watch reward_history vs extrinsic_reward_history to confirm.
replay_coef = 0.1        # self-imitation weight -- see update_replay_buffer/self_imitation_loss.
                          # H1-H4 ablation showed budget/curiosity alone don't retain a rare win;
                          # this is the mechanism meant to actually fix that (DESIGN_LOG 2026-08-16).
replay_batch = 4          # chunks replayed per update
replay_buffer_size = 8    # top-K best chunks ever seen, kept across the whole run

In [ ]:
reward_history = []
extrinsic_reward_history = []
grpo_loss_history = []
jepa_loss_history = []
entropy_history = []
replay_loss_history = []
best_run = {"reward": float("-inf"), "trajectory": None}
replay_buffer = []

for step in range(epochs):
    rollouts = collect_group(env, model, group_size, t_horizon, max_chunks, curiosity_coef)
    grpo_loss, jepa_loss, entropy = compute_losses(rollouts, model)
    replay_loss = self_imitation_loss(model, replay_buffer, replay_batch)
    # Subtracting entropy_coef * entropy pushes the optimizer to *maximize*
    # entropy (keep the action distribution spread out) alongside minimizing
    # the other two losses -- see entropy_coef comment above for why.
    loss = grpo_loss + jepa_weight * jepa_loss - entropy_coef * entropy + replay_coef * replay_loss

    optimizer.zero_grad()
    loss.backward()
    # Trust-region safety net: caps how far a single update can move the
    # weights, regardless of cause. Needed because grpo_loss can spike
    # arbitrarily -- e.g. when a sampled action turns out illegal past step 0
    # (only step 0's legality is masked, see DESIGN_LOG risk #3) and gets
    # replaced by a random legal action, its log_prob under an increasingly
    # confident policy can be extremely negative, blowing up the loss even
    # though no real reward signal justifies the update (observed: grpo_loss
    # went -64 -> -1688 over 490 steps while mean_reward stayed flat).
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    model.update_target()  # EMA update of target_eyes, after the online eyes step

    best_run = update_best_run(best_run, rollouts)
    replay_buffer = update_replay_buffer(replay_buffer, rollouts, replay_buffer_size)

    # Tracked separately (see DESIGN_LOG 2026-08-16): mean_reward mixes in the
    # curiosity bonus, extrinsic_mean_reward is task-only -- watch the latter
    # to judge real progress, the former just confirms exploration is happening.
    mean_reward = sum(sum(c["reward"] for c in chunks) for chunks, _ in rollouts) / len(rollouts)
    extrinsic_mean_reward = sum(sum(c["extrinsic_reward"] for c in chunks) for chunks, _ in rollouts) / len(rollouts)
    reward_history.append(mean_reward)
    extrinsic_reward_history.append(extrinsic_mean_reward)
    grpo_loss_history.append(grpo_loss.item())
    jepa_loss_history.append(jepa_loss.item())
    entropy_history.append(entropy.item())
    replay_loss_history.append(replay_loss.item())

    if step % 10 == 0:
        print(f"step {step:4d} | grpo_loss {grpo_loss.item():8.4f} | jepa_loss {jepa_loss.item():7.4f} | entropy {entropy.item():6.3f} | replay_loss {replay_loss.item():7.3f} | reward(total) {mean_reward:7.3f} | reward(task) {extrinsic_mean_reward:7.3f} | best(task) {best_run['reward']:7.3f}")

fig, axes = plt.subplots(1, 5, figsize=(24, 4))
axes[0].plot(reward_history, label="total (task+curiosity)")
axes[0].plot(extrinsic_reward_history, label="task only")
axes[0].set_xlabel("update"); axes[0].set_ylabel("mean episode reward"); axes[0].legend()
axes[1].plot(grpo_loss_history); axes[1].set_xlabel("update"); axes[1].set_ylabel("grpo_loss")
axes[2].plot(jepa_loss_history); axes[2].set_xlabel("update"); axes[2].set_ylabel("jepa_loss")
axes[3].plot(entropy_history); axes[3].set_xlabel("update"); axes[3].set_ylabel("policy entropy")
axes[4].plot(replay_loss_history); axes[4].set_xlabel("update"); axes[4].set_ylabel("replay_loss (self-imitation)")
plt.tight_layout()
plt.show()

In [29]:
def count_parameters(module):
    return sum(p.numel() for p in module.parameters() if p.requires_grad)

print(f"{'eyes':<12} {count_parameters(model.eyes):>10,}")
print(f"{'brain':<12} {count_parameters(model.brain):>10,}")
print(f"{'intuition':<12} {count_parameters(model.intuition):>10,}")
print(f"{'-' * 12} {'-' * 10}")
print(f"{'total':<12} {count_parameters(model):>10,}")
print(f"({'target_eyes (frozen EMA copy, excluded above)':<12}: {sum(p.numel() for p in model.target_eyes.parameters()):,})")

eyes              3,824
brain            69,496
intuition        69,184
------------ ----------
total           142,504
(target_eyes (frozen EMA copy, excluded above): 3,824)


In [30]:
# Watch the single best episode seen across the whole training run -- use this
# to inspect what the agent has actually learned (does it head for the cross?
# does it trigger rotations on purpose or by accident?).
# best_run["reward"] is task-only (extrinsic); best_run["total_reward"] includes
# the curiosity bonus too, shown for comparison (see DESIGN_LOG 2026-08-16).
print(f"best episode reward: task={best_run['reward']:.3f}  total={best_run.get('total_reward', float('nan')):.3f}  ({len(best_run['trajectory'])} frames)")
render_trajectory(best_run["trajectory"])

best episode reward: task=-0.480  total=-0.480  (49 frames)


# Self-imitation A/B test

Launches two background `src/train.py` runs, identical except `replay_coef`
(0.0 vs 0.1) -- same seed, so the only difference is whether the self-imitation
buffer (DESIGN_LOG 2026-08-16) is active. Answers: does replaying past
successes actually beat plain GRPO, after the H1-H4 ablation showed epochs
budget and curiosity_coef alone don't retain a rare win?

In [ ]:
import subprocess

ab_runs = [
    dict(run_name="replay_off", replay_coef=0.0),
    dict(run_name="replay_on", replay_coef=0.1),
]

ab_procs = []
for r in ab_runs:
    cmd = [
        "python3", "../src/train.py",
        "--run-name", r["run_name"],
        "--epochs", "500",
        "--group-size", "24",
        "--curiosity-coef", "0.0",   # isolate replay's effect -- H1-H4 showed curiosity adds no measurable gain
        "--entropy-coef", "0.001",
        "--replay-coef", str(r["replay_coef"]),
        "--seed", "0",
        "--output-dir", "../results",
    ]
    run_env = os.environ.copy()
    run_env["OMP_NUM_THREADS"] = "6"  # 2 processes x 6 threads = 12 cores, no oversubscription
    run_env["MKL_NUM_THREADS"] = "6"
    log_file = open(f"../results/{r['run_name']}.log", "w")
    proc = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT, env=run_env)
    ab_procs.append((r["run_name"], proc, log_file))
    print(f"launched {r['run_name']} (replay_coef={r['replay_coef']}, pid {proc.pid})")

print("Running in background -- run the next cell to block until both finish and compare.")

In [ ]:
import json

for name, proc, log_file in ab_procs:
    proc.wait()
    log_file.close()
    print(f"{name} finished (exit code {proc.returncode})")

print()
for r in ab_runs:
    with open(f"../results/{r['run_name']}.json") as f:
        d = json.load(f)
    ext = d["extrinsic_reward_history"]
    n_pos = sum(1 for v in ext if v > -0.48 + 1e-6)  # epochs where the group beat the always-fail baseline
    print(f"{r['run_name']:<12} replay_coef={r['replay_coef']:<5} "
          f"best_task={d['best_extrinsic_reward']:+.3f}  epochs_with_progress={n_pos}/{len(ext)}  "
          f"elapsed={d['elapsed_seconds']:.0f}s")